# Depth Anything v2 — Validation Notebook

This notebook validates the exported ONNX depth model against laser-measured ground-truth distances.

**Steps:**
1. Load ONNX model (FP32 or INT8)
2. Run inference on test images
3. Apply pig segmentation mask to isolate the pig region
4. Convert relative depth → metric using a calibration scale factor
5. Compute MAE / RMSE vs laser ground truth
6. Plot error distribution

**Target accuracy:** MAE < 15 cm vs laser measurement

In [ ]:
# Install dependencies if needed
# !pip install onnxruntime opencv-python pillow numpy matplotlib

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import onnxruntime as ort

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
ONNX_MODEL_PATH = "models/depth_anything_v2_small_int8.onnx"  # or _small.onnx for FP32

# Test dataset: list of dicts with keys:
#   image_path   : str  — path to image file
#   laser_dist_m : float — laser-measured camera-to-pig distance in metres
#   mask_path    : str | None — path to binary mask PNG (255=pig, 0=background)
#                              if None, the whole image is used
TEST_DATASET = [
    # Example entries — replace with your actual data
    # {"image_path": "data/pig_001.jpg", "laser_dist_m": 1.35, "mask_path": "data/pig_001_mask.png"},
    # {"image_path": "data/pig_002.jpg", "laser_dist_m": 0.87, "mask_path": None},
]

# Input size the model was exported with
INPUT_SIZE = 518  # change to 256 if you used --input-size 256 during export

# ── Calibration scale factor ───────────────────────────────────────────────
# Depth Anything v2 outputs *relative* inverse depth (arbitrary units).
# To convert to metric we need to scale: metric_depth = depth_raw * SCALE_FACTOR
# This is estimated from the calibration pass below; set manually if known.
SCALE_FACTOR = None  # will be computed automatically from the dataset

## 2. Load ONNX Model

In [ ]:
if not Path(ONNX_MODEL_PATH).exists():
    raise FileNotFoundError(
        f"Model not found: {ONNX_MODEL_PATH}\n"
        "Run export_to_onnx.py first:\n"
        "  python export_to_onnx.py --input-size 518 --quantize"
    )

sess = ort.InferenceSession(
    ONNX_MODEL_PATH,
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"],
)
inp_name = sess.get_inputs()[0].name
print(f"Model loaded: {ONNX_MODEL_PATH}")
print(f"Input name  : {inp_name}")
print(f"Input shape : {sess.get_inputs()[0].shape}")
print(f"Output name : {sess.get_outputs()[0].name}")
print(f"Output shape: {sess.get_outputs()[0].shape}")

## 3. Preprocessing Helper

In [ ]:
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def preprocess(image_path: str, size: int) -> tuple[np.ndarray, tuple[int, int]]:
    """Load image, resize, normalise → (1,3,H,W) float32.  Returns tensor + original (H,W)."""
    img = Image.open(image_path).convert("RGB")
    orig_size = (img.height, img.width)
    img = img.resize((size, size), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = (arr - MEAN) / STD
    arr = arr.transpose(2, 0, 1)[np.newaxis]  # (1,3,H,W)
    return arr, orig_size


def infer(tensor: np.ndarray) -> np.ndarray:
    """Run ONNX inference, return (H,W) depth map."""
    out = sess.run(None, {inp_name: tensor})
    depth = out[0]  # shape varies: (1,H,W) or (H,W)
    if depth.ndim == 3:
        depth = depth[0]
    return depth  # float32, relative inverse depth


def load_mask(mask_path: str | None, orig_size: tuple[int, int]) -> np.ndarray | None:
    """Load binary mask and resize to orig_size (H,W).  Returns bool array or None."""
    if mask_path is None:
        return None
    mask = Image.open(mask_path).convert("L")
    mask = mask.resize((orig_size[1], orig_size[0]), Image.NEAREST)
    return np.array(mask) > 127


def median_depth_in_mask(depth_map: np.ndarray, mask: np.ndarray | None) -> float:
    """Return median depth value inside the mask (or whole image if mask is None)."""
    if mask is not None:
        # Resize depth to mask resolution
        d_resized = cv2.resize(depth_map, (mask.shape[1], mask.shape[0]), interpolation=cv2.INTER_LINEAR)
        values = d_resized[mask]
    else:
        values = depth_map.flatten()
    return float(np.median(values))

## 4. Calibration — Estimate Scale Factor

Since Depth Anything v2 returns *relative* depth, we compute a linear scale factor:

```
scale = median(laser_dist_m / raw_depth_median)  over all calibration samples
```

In [ ]:
if len(TEST_DATASET) == 0:
    print("⚠️  TEST_DATASET is empty — add image paths and laser distances above.")
    print("   Generating a synthetic demo instead …")

    # ── Synthetic demo (no real images needed) ────────────────────────────
    np.random.seed(42)
    N = 10
    fake_raw_depths = np.random.uniform(0.3, 0.9, N)
    fake_laser = fake_raw_depths * 1.8 + np.random.normal(0, 0.05, N)  # scale=1.8 + noise
    scale = float(np.median(fake_laser / fake_raw_depths))
    predicted = fake_raw_depths * scale
    errors_m = predicted - fake_laser
    print(f"  [Synthetic demo] scale factor = {scale:.4f}")
    demo_mode = True
else:
    demo_mode = False
    raw_depths = []
    laser_dists = []

    for entry in TEST_DATASET:
        tensor, orig_size = preprocess(entry["image_path"], INPUT_SIZE)
        depth_map = infer(tensor)
        mask = load_mask(entry.get("mask_path"), orig_size)
        raw_d = median_depth_in_mask(depth_map, mask)
        raw_depths.append(raw_d)
        laser_dists.append(entry["laser_dist_m"])
        print(f"  {Path(entry['image_path']).name:30s}  raw={raw_d:.4f}  laser={entry['laser_dist_m']:.3f} m")

    raw_depths = np.array(raw_depths, dtype=np.float64)
    laser_dists = np.array(laser_dists, dtype=np.float64)

    if SCALE_FACTOR is None:
        scale = float(np.median(laser_dists / raw_depths))
        print(f"\nCalibration scale factor (median): {scale:.4f}")
    else:
        scale = SCALE_FACTOR
        print(f"Using manual scale factor: {scale:.4f}")

    predicted = raw_depths * scale
    errors_m = predicted - laser_dists

## 5. Accuracy Metrics

In [ ]:
mae  = float(np.mean(np.abs(errors_m)))
rmse = float(np.sqrt(np.mean(errors_m ** 2)))
max_err = float(np.max(np.abs(errors_m)))

print("=" * 40)
print(f"  MAE  : {mae * 100:.1f} cm")
print(f"  RMSE : {rmse * 100:.1f} cm")
print(f"  Max  : {max_err * 100:.1f} cm")
print("=" * 40)

TARGET_MAE_M = 0.15  # 15 cm
if mae <= TARGET_MAE_M:
    print(f"  ✅  MAE {mae*100:.1f} cm ≤ target {TARGET_MAE_M*100:.0f} cm")
else:
    print(f"  ❌  MAE {mae*100:.1f} cm > target {TARGET_MAE_M*100:.0f} cm — needs improvement")

## 6. Error Distribution Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of errors
axes[0].hist(errors_m * 100, bins=20, color="steelblue", edgecolor="white", alpha=0.85)
axes[0].axvline(0, color="black", linestyle="--", linewidth=1)
axes[0].axvline(mae * 100, color="red", linestyle="--", linewidth=1, label=f"MAE = {mae*100:.1f} cm")
axes[0].axvline(-mae * 100, color="red", linestyle="--", linewidth=1)
axes[0].set_xlabel("Error (cm)")
axes[0].set_ylabel("Count")
axes[0].set_title("Depth Error Distribution")
axes[0].legend()

# Predicted vs ground-truth scatter (skip in pure demo mode)
if not demo_mode:
    axes[1].scatter(laser_dists, predicted, alpha=0.75, color="steelblue")
    lo, hi = min(laser_dists.min(), predicted.min()), max(laser_dists.max(), predicted.max())
    axes[1].plot([lo, hi], [lo, hi], "k--", linewidth=1, label="Perfect")
    axes[1].set_xlabel("Laser distance (m)")
    axes[1].set_ylabel("Predicted distance (m)")
    axes[1].set_title("Predicted vs Ground Truth")
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "No real data —\nrun with actual images",
                 ha="center", va="center", fontsize=12, color="grey",
                 transform=axes[1].transAxes)
    axes[1].set_title("Predicted vs Ground Truth")

fig.suptitle("Depth Anything v2 — Validation Results", fontsize=13)
fig.tight_layout()
plt.savefig("validation_results.png", dpi=120)
plt.show()
print("Saved: validation_results.png")

## 7. Depth Map Visualisation (Single Image)

In [ ]:
# Change this path to any of your test images
VIS_IMAGE = TEST_DATASET[0]["image_path"] if TEST_DATASET else None

if VIS_IMAGE and Path(VIS_IMAGE).exists():
    tensor, orig_size = preprocess(VIS_IMAGE, INPUT_SIZE)
    depth_map = infer(tensor)
    depth_norm = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-8)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(Image.open(VIS_IMAGE))
    axes[0].set_title("Input image")
    axes[0].axis("off")

    im = axes[1].imshow(depth_norm, cmap="inferno")
    axes[1].set_title("Predicted depth map")
    axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04, label="Relative depth (normalised)")
    fig.tight_layout()
    plt.show()
else:
    print("⚠️  No test image available for visualisation.")
    print("   Set VIS_IMAGE to a valid image path and re-run this cell.")

## 8. Summary Table

In [ ]:
import pandas as pd  # optional — only used here

if not demo_mode and len(TEST_DATASET) > 0:
    rows = []
    for i, entry in enumerate(TEST_DATASET):
        rows.append({
            "Image": Path(entry["image_path"]).name,
            "Laser (m)": f"{entry['laser_dist_m']:.3f}",
            "Predicted (m)": f"{predicted[i]:.3f}",
            "Error (cm)": f"{errors_m[i] * 100:+.1f}",
            "|Error| (cm)": f"{abs(errors_m[i]) * 100:.1f}",
        })
    df = pd.DataFrame(rows)
    display(df)
    print(f"\nScale factor used : {scale:.4f}")
    print(f"MAE              : {mae*100:.1f} cm")
    print(f"RMSE             : {rmse*100:.1f} cm")
else:
    print("Add real test images to TEST_DATASET to see the comparison table.")